# Spark Setup

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

Streaming Implementation

In [ ]:

# ==================================================
# Schema
# ==================================================

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])


# ==================================================
# Kafka reader
# ==================================================

def read_topic(topic, source):
    return (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            f"{HOST_IP}:9092"
        )
        .option("subscribe", topic)
        .load()
        .selectExpr("CAST(value AS STRING)")
        .withColumn("source", lit(source))
        .select(
            from_json(
                col("value"),
                event_schema
            ).alias("data"),
            col("source")
        )
        .select("data.*", "source")
        .withColumn(
            "event_time",
            to_timestamp(col("timestamp"))
        )
    )


# ==================================================
# Read streams
# ==================================================

camera_a_stream = read_topic(
    "camera-events-A",
    "camera-a"
)

camera_b_stream = read_topic(
    "camera-events-B",
    "camera-b"
)

camera_c_stream = read_topic(
    "camera-events-C",
    "camera-c"
)


# ==================================================
# Combine streams
# ==================================================

combined_stream = (
    camera_a_stream
    .union(camera_b_stream)
    .union(camera_c_stream)
    .withWatermark(
        "event_time",
        "10 minutes"
    )
)


# ==================================================
# Camera metadata
# ==================================================

camera_df = spark.read.csv(
    f"{Path('..')}/data/camera.csv",
    header=True,
    inferSchema=True
)


# ==================================================
# Enrich stream
# ==================================================

events_with_camera = (
    combined_stream
    .join(camera_df, "camera_id")
)


# ==================================================
# Instant violations
# ==================================================

instant_violations = (
    events_with_camera
    .filter(
        col("speed_reading")
        > col("speed_limit")
    )
)


# ==================================================
# Average speed join logic
# ==================================================

start_events = (
    events_with_camera.alias("start")
)

end_events = (
    events_with_camera.alias("end")
)

joined_stream = (
    start_events.join(
        end_events,
        expr("""
            start.car_plate = end.car_plate
            AND start.position < end.position
            AND start.event_time < end.event_time
            AND end.event_time <=
                start.event_time + interval 10 minutes
        """)
    )
)


# ==================================================
# Average speed calculation
# ==================================================

average_violations = (
    joined_stream
    .withColumn(
        "distance_km",
        abs(
            col("end.position")
            - col("start.position")
        )
    )
    .withColumn(
        "travel_time_hours",
        (
            unix_timestamp(
                col("end.event_time")
            )
            -
            unix_timestamp(
                col("start.event_time")
            )
        ) / 3600
    )
    .withColumn(
        "average_speed",
        col("distance_km")
        / col("travel_time_hours")
    )
    .filter(
        col("average_speed")
        > col("end.speed_limit")
    )
)


# ==================================================
# Logger helper
# ==================================================

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger


# ==================================================
# Queries
# ==================================================

combined_query = (
    combined_stream.writeStream
    .foreachBatch(
        log_batch("combined_stream processed")
    )
    .outputMode("append")
    .start()
)

instant_query = (
    instant_violations.writeStream
    .foreachBatch(
        log_batch(
            "instant_violations processed"
        )
    )
    .outputMode("append")
    .start()
)

average_query = (
    average_violations.writeStream
    .foreachBatch(
        log_batch(
            "average_violations processed"
        )
    )
    .outputMode("append")
    .start()
)


# ==================================================
# Wait for all queries
# ==================================================

spark.streams.awaitAnyTermination()


[2026-05-11 08:52:07] combined_stream processed
Spark Batch ID: 0
Rows received: 0
+--------+--------+---------+---------+---------+-------------+------+----------+
|event_id|batch_id|car_plate|camera_id|timestamp|speed_reading|source|event_time|
+--------+--------+---------+---------+---------+-------------+------+----------+
+--------+--------+---------+---------+---------+-------------+------+----------+


[2026-05-11 08:52:07] instant_violations processed
Spark Batch ID: 0
Rows received: 0
+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|event_time|latitude|longitude|position|speed_limit|
+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+
+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+


[2026-05-1


[2026-05-11 08:52:29] combined_stream processed
Spark Batch ID: 3
Rows received: 24
+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+-----------+-----------------+-------------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|event_time|latitude|longitude|position|speed_limit|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|event_time|latitude|longitude|position|speed_limit|distance_km|travel_time_hours|average_speed|
+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+---------+--------+--------+---------+---------+-------------+------+----------+--------+---------+--------+-----------+-----------+-----------------+-------------+
+---------+--------+--------+---------+---------+--------


[2026-05-11 08:52:33] average_violations processed
Spark Batch ID: 1
Rows received: 2

[2026-05-11 08:52:42] instant_violations processed
Spark Batch ID: 4
Rows received: 11
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+--------------------------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |event_time                |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+--------------------------+-----------+-----------+--------+-----------+
|1        |26c4d7ff-1411-4a00-b418-3e08554e3cfa|26      |SSP 248  |2024-01-01T11:13:08       |135.4        |camera-a|2024-01-01 11:13:08       |2.157730731|102.6601002|152.5   |110        |
|1        |69bafd7e-3447-41c0-b2c0-f3fb344c3c44|26      |FL 80   

+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+--------------------------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |event_time                |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+--------------------------+-----------+-----------+--------+-----------+
|1        |07f67ff2-3e1d-4aac-8b99-4f8e044abf86|27      |WU 4     |2024-01-01T11:21:50       |158.2        |camera-a|2024-01-01 11:21:50       |2.157730731|102.6601002|152.5   |110        |
|1        |35ae3639-35d7-4a99-9ccd-23f2bf47a158|27      |CD 8184  |2024-01-01T11:21:50       |121.2        |camera-a|2024-01-01 11:21:50       |2.157730731|102.6601002|152.5   |110        |
|1        |1039582d-555a-43ee-bb88-6c4f3c27d2da|27

+------------------------------------+--------+---------+---------+-------------------+-------------+--------+-------------------+
|event_id                            |batch_id|car_plate|camera_id|timestamp          |speed_reading|source  |event_time         |
+------------------------------------+--------+---------+---------+-------------------+-------------+--------+-------------------+
|e4de79bb-c089-4288-b5ea-7125d1985dc7|30      |RTI 4    |1        |2024-01-01T11:39:59|92.8         |camera-a|2024-01-01 11:39:59|
|4a7777db-177d-4a4f-b5f3-df9b00d50886|30      |UUK 911  |1        |2024-01-01T11:40:00|63.6         |camera-a|2024-01-01 11:40:00|
|59921734-4cc2-4525-8fd7-8e8b110826c9|30      |OEQ 64   |1        |2024-01-01T11:40:03|151.1        |camera-a|2024-01-01 11:40:03|
|c7f07c28-2d98-4850-9c2b-00455dd7e242|30      |AZH 16   |1        |2024-01-01T11:40:04|77.6         |camera-a|2024-01-01 11:40:04|
|44642a92-b9d8-4090-89a3-0104be365fac|30      |VDL 6    |1        |2024-01-01T11:40

+---------+------------------------------------+--------+---------+-------------------+-------------+--------+-------------------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp          |speed_reading|source  |event_time         |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+-------------------+-------------+--------+-------------------+-----------+-----------+--------+-----------+
|1        |d790ffff-a780-4672-b2c4-2146aaeb0e22|31      |SH 202   |2024-01-01T11:49:40|142.5        |camera-a|2024-01-01 11:49:40|2.157730731|102.6601002|152.5   |110        |
|1        |880f8d26-3ad3-4057-ac6f-840a65ec7adf|31      |QS 64    |2024-01-01T11:49:41|146.7        |camera-a|2024-01-01 11:49:41|2.157730731|102.6601002|152.5   |110        |
|1        |e10411ab-b16b-4c91-b249-13a90a607432|31      |AC 6862  |2024-01-01T11:49:43|146.2        |camera-a|2024-01-01


[2026-05-11 08:53:11] combined_stream processed
Spark Batch ID: 11
Rows received: 42
+------------------------------------+--------+---------+---------+-------------------+-------------+--------+-------------------+
|event_id                            |batch_id|car_plate|camera_id|timestamp          |speed_reading|source  |event_time         |
+------------------------------------+--------+---------+---------+-------------------+-------------+--------+-------------------+
|f95f734d-27b8-4c51-9c4a-5e788bd9d2f8|33      |MKA 437  |1        |2024-01-01T12:04:41|74.5         |camera-a|2024-01-01 12:04:41|
|88569824-f9fe-4479-a150-8d9bc1e7a09c|33      |BS 80    |1        |2024-01-01T12:04:40|97.3         |camera-a|2024-01-01 12:04:40|
|d6e424d1-9cc1-41f6-ab95-d4b74fde789e|33      |KC 1     |1        |2024-01-01T12:04:37|90.9         |camera-a|2024-01-01 12:04:37|
|3c652b11-276c-476d-9652-daadae41632e|33      |QK 91    |1        |2024-01-01T12:04:41|77.2         |camera-a|2024-01-01 12:04:4

+---------+------------------------------------+--------+---------+-------------------+-------------+--------+-------------------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp          |speed_reading|source  |event_time         |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+-------------------+-------------+--------+-------------------+-----------+-----------+--------+-----------+
|1        |afa6995e-49c3-4333-bdab-415448d83263|35      |XQ 471   |2024-01-01T12:19:51|138.5        |camera-a|2024-01-01 12:19:51|2.157730731|102.6601002|152.5   |110        |
|1        |97e8fe50-f378-4f4f-b6a6-7e65f0b58fff|35      |MY 557   |2024-01-01T12:19:52|150.6        |camera-a|2024-01-01 12:19:52|2.157730731|102.6601002|152.5   |110        |
|1        |11adf5ca-259c-40cb-b3d9-0bc3b222b1fb|35      |VYM 40   |2024-01-01T12:19:49|119.8        |camera-a|2024-01-01